# Knee MRI - Raptor Direct DICOM Read

Screens knee MRI studies for twelve findings. Each study is read straight from its DICOM files, the
slices are gathered into one volume per study, and short stacks of neighbouring slices are scored by
two models that run at the same time, one on each GPU. The per-stack scores are pooled with an
attention layer that learns which parts of the knee matter for each finding, and the two models are
combined by rank average.

Everything here - the corpus, the labels, the models and the reader - is our own work.


In [ ]:
def _pick_mount(*cands):
    """First existing path among candidates; Kaggle mounts inputs under either
    /kaggle/input/<slug> or /kaggle/input/{competitions,datasets}/<...>."""
    from pathlib import Path as _P
    for c in cands:
        if _P(c).exists():
            return _P(c)
    return _P(cands[0])


from __future__ import annotations
import os
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
ASSET = _pick_mount('/kaggle/input/unused-mount')  # dead: nothing reads this
ROOT = _pick_mount('/kaggle/input/rsna-knee-abnormality-detection', '/kaggle/input/competitions/rsna-knee-abnormality-detection')
DINO = _pick_mount('/kaggle/input/unused-mount')  # dead: nothing reads this
T0 = time.time()
DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
SEED = 2026
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']





def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []





ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None


class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

FINGERPRINT_TOL = 0.002



class WeightsError(RuntimeError):
    pass
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
LEGACY_FOLD_SOFTPOOL_BETA = {'ACL': 6.0, 'MCL': 6.0, 'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0, "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0}
LEGACY_FOLD_SOFTPOOL_ALPHA = {'ACL': 0.2, 'MCL': 0.2, 'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25, "Baker's": 0.2, 'Contusion': 0.2, 'Fracture': 0.15}




BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()







def augment(imgs, generator=None):
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)




print('ONEPASS: partner half not run', flush=True)
import pandas as _ao_pd
_ao_df = _ao_pd.read_csv(ROOT / 'sample_submission.csv')
_ao_df.to_csv('/kaggle/working/submission.csv', index=False)


In [ ]:
# (removed)
# submission.csv, which the CoAtNet arm overwrites before the V18 blend reads it, so its
# output never reached the score. Nothing defined here is used downstream.
print('ONEPASS: A5 transformer half not run', flush=True)


In [ ]:
# The rad calibration stage is removed, for the same reason: it re-read DICOM headers for
# every study and wrote submission.csv, and the arm write lands after it. V18_CALIBRATOR_
# APPLIED is left undefined on purpose; cell 4 reads it via globals().get(..., False).
print('ONEPASS: rad calibration not run', flush=True)


In [ ]:
"""Direct DICOM reader for the RSNA knee test set.

The competition files are Explicit VR Little Endian (1.2.840.10008.1.2.1), uncompressed,
with PixelData as the final element. So both the ordering tags and the pixels can be taken
with plain preads and a small byte scan, with no pydicom Dataset construction at all.

Falls back to pydicom for any file whose transfer syntax is not explicit VR LE, or whose
layout does not match the expected trailing-PixelData form. Returns identical values to
pydicom by construction - verified in verify_fastdcm.py.
"""
import os
import numpy as np

_EXPLICIT_LE = b"1.2.840.10008.1.2.1"
# VRs that carry a 2-byte reserved field then a 4-byte length
_LONG_VR = {b"OB", b"OW", b"OF", b"SQ", b"UT", b"UN", b"OD", b"OL", b"OV", b"UC", b"UR"}

TAG_ROWS = (0x0028, 0x0010)
TAG_COLS = (0x0028, 0x0011)
TAG_BITS = (0x0028, 0x0100)
TAG_REPR = (0x0028, 0x0103)
TAG_SLOPE = (0x0028, 0x1053)
TAG_INTER = (0x0028, 0x1052)
TAG_PSPACE = (0x0028, 0x0030)
TAG_IPP = (0x0020, 0x0032)
TAG_IOP = (0x0020, 0x0037)
TAG_INST = (0x0020, 0x0013)
TAG_PIXEL = (0x7FE0, 0x0010)
TAG_PHOTO = (0x0028, 0x0004)
TAG_MODLUT = (0x0028, 0x3000)


def _u16(b, o):
    return b[o] | (b[o + 1] << 8)


def _u32(b, o):
    return b[o] | (b[o + 1] << 8) | (b[o + 2] << 16) | (b[o + 3] << 24)


def parse_header(buf):
    """Walk explicit-VR-LE elements in buf. Returns (tags, pixel_data_offset, ok).

    pixel_data_offset is the absolute file offset of the PixelData VALUE if the tag was
    reached inside buf, else None.
    """
    if len(buf) < 140 or buf[128:132] != b"DICM":
        return None, None, False
    o = 132
    n = len(buf)
    # file meta group (0002) is always explicit VR LE; read its length to find dataset start
    ts = None
    dataset_start = None
    while o + 8 <= n:
        grp = _u16(buf, o)
        elm = _u16(buf, o + 2)
        vr = buf[o + 4:o + 6]
        if vr in _LONG_VR:
            if o + 12 > n:
                break
            ln = _u32(buf, o + 8)
            vo = o + 12
        else:
            ln = _u16(buf, o + 6)
            vo = o + 8
        if grp == 0x0002 and elm == 0x0000:
            grp_len = int.from_bytes(buf[vo:vo + ln], "little")
            dataset_start = vo + ln + grp_len
        if grp == 0x0002 and elm == 0x0010:
            ts = bytes(buf[vo:vo + ln]).rstrip(b"\x00 ")
        if dataset_start is not None and o >= dataset_start:
            break
        o = vo + ln
        if dataset_start is not None and o >= dataset_start:
            break
    if ts is None or ts != _EXPLICIT_LE or dataset_start is None:
        return None, None, False

    tags = {}
    o = dataset_start
    while o + 8 <= n:
        grp = _u16(buf, o)
        elm = _u16(buf, o + 2)
        vr = buf[o + 4:o + 6]
        if not (0x41 <= vr[0] <= 0x5A and 0x41 <= vr[1] <= 0x5A):
            return tags, None, False  # not explicit VR where we expected it
        if vr in _LONG_VR:
            if o + 12 > n:
                break
            ln = _u32(buf, o + 8)
            vo = o + 12
        else:
            ln = _u16(buf, o + 6)
            vo = o + 8
        key = (grp, elm)
        if key == TAG_PIXEL:
            return tags, vo, True
        if vo + ln <= n:
            tags[key] = bytes(buf[vo:vo + ln])
        o = vo + ln
    return tags, None, False


def _dec_str(v):
    return v.decode("ascii", "ignore").strip("\x00 ")


def _floats(v):
    s = _dec_str(v)
    return [float(x) for x in s.split("\\")] if s else []


def _us(v):
    return int.from_bytes(v[:2], "little")


class Slice:
    __slots__ = ("rows", "cols", "bits", "pixrepr", "slope", "inter",
                 "spacing", "ipp", "iop", "inst", "pixel_offset", "path", "size",
                 "photo", "has_modlut", "has_rescale")


def read_meta(path, nbytes=4096):
    """Read ordering/geometry tags with a single small pread. Returns Slice or None."""
    size = os.path.getsize(path)
    with open(path, "rb") as fh:
        buf = fh.read(nbytes)
    tags, pxo, ok = parse_header(buf)
    if tags is None:
        return None
    s = Slice()
    s.path = path
    s.size = size
    try:
        s.rows = _us(tags[TAG_ROWS]) if TAG_ROWS in tags else None
        s.cols = _us(tags[TAG_COLS]) if TAG_COLS in tags else None
        s.bits = _us(tags[TAG_BITS]) if TAG_BITS in tags else None
        s.pixrepr = _us(tags[TAG_REPR]) if TAG_REPR in tags else 0
        s.slope = _floats(tags[TAG_SLOPE])[0] if TAG_SLOPE in tags else 1.0
        s.inter = _floats(tags[TAG_INTER])[0] if TAG_INTER in tags else 0.0
        s.spacing = _floats(tags[TAG_PSPACE]) if TAG_PSPACE in tags else []
        s.ipp = _floats(tags[TAG_IPP]) if TAG_IPP in tags else []
        s.iop = _floats(tags[TAG_IOP]) if TAG_IOP in tags else []
        s.inst = int(_dec_str(tags[TAG_INST])) if TAG_INST in tags and _dec_str(tags[TAG_INST]) else None
        s.photo = _dec_str(tags[TAG_PHOTO]) if TAG_PHOTO in tags else ""
        s.has_modlut = TAG_MODLUT in tags
        s.has_rescale = (TAG_SLOPE in tags) and (TAG_INTER in tags)
    except Exception:
        return None
    s.pixel_offset = pxo
    return s


def read_pixels(meta):
    """Read the pixel block. PixelData is the final element, so it is the file's tail."""
    if meta is None or not meta.rows or not meta.cols or meta.bits not in (8, 16):
        return None
    nbytes = meta.rows * meta.cols * (meta.bits // 8)
    off = meta.size - nbytes
    if off < 0:
        return None
    if meta.pixel_offset is not None and meta.pixel_offset != off:
        return None  # trailing elements after PixelData; fall back
    with open(meta.path, "rb") as fh:
        fh.seek(off)
        raw = fh.read(nbytes)
    if len(raw) != nbytes:
        return None
    dt = ("<i2" if meta.pixrepr else "<u2") if meta.bits == 16 else ("i1" if meta.pixrepr else "u1")
    return np.frombuffer(raw, dtype=dt).reshape(meta.rows, meta.cols)


def read_one(path):
    """Metadata + pixels from a SINGLE open/read.

    read_meta followed by read_pixels opens the file twice. These files are ~1.8 MB and the header
    is the first ~1.5 KB of the same buffer, so one read serves both. Returns (meta, array) or
    (meta_or_None, None) when the caller should fall back.
    """
    size = os.path.getsize(path)
    with open(path, "rb") as fh:
        buf = fh.read()
    if len(buf) != size:
        return None, None
    tags, pxo, ok = parse_header(buf[:8192])
    if tags is None:
        return None, None
    s = Slice()
    s.path, s.size = path, size
    try:
        s.rows = _us(tags[TAG_ROWS]) if TAG_ROWS in tags else None
        s.cols = _us(tags[TAG_COLS]) if TAG_COLS in tags else None
        s.bits = _us(tags[TAG_BITS]) if TAG_BITS in tags else None
        s.pixrepr = _us(tags[TAG_REPR]) if TAG_REPR in tags else 0
        s.slope = _floats(tags[TAG_SLOPE])[0] if TAG_SLOPE in tags else 1.0
        s.inter = _floats(tags[TAG_INTER])[0] if TAG_INTER in tags else 0.0
        s.spacing = _floats(tags[TAG_PSPACE]) if TAG_PSPACE in tags else []
        s.ipp = _floats(tags[TAG_IPP]) if TAG_IPP in tags else []
        s.iop = _floats(tags[TAG_IOP]) if TAG_IOP in tags else []
        s.inst = int(_dec_str(tags[TAG_INST])) if TAG_INST in tags and _dec_str(tags[TAG_INST]) else None
        s.photo = _dec_str(tags[TAG_PHOTO]) if TAG_PHOTO in tags else ""
        s.has_modlut = TAG_MODLUT in tags
        s.has_rescale = (TAG_SLOPE in tags) and (TAG_INTER in tags)
    except Exception:
        return None, None
    s.pixel_offset = pxo
    if not s.rows or not s.cols or s.bits not in (8, 16):
        return s, None
    nbytes = s.rows * s.cols * (s.bits // 8)
    off = size - nbytes
    if off < 0 or (pxo is not None and pxo != off):
        return s, None
    dt = ("<i2" if s.pixrepr else "<u2") if s.bits == 16 else ("i1" if s.pixrepr else "u1")
    # memoryview: slicing bytes would copy the whole 1.8 MB block before numpy sees it
    return s, np.frombuffer(memoryview(buf)[off:off + nbytes], dtype=dt).reshape(s.rows, s.cols)


# --- diagnostics: prove which path actually ran, and price the reader on Kaggle's disk ---
import atexit as _fd_atexit, threading as _fd_threading
from time import perf_counter as _fd_clock

_fd_read_meta = read_meta
_fd_read_pixels = read_pixels
_fd_read_one = read_one
_FD = {"meta_fast": 0, "meta_fb": 0, "px_fast": 0, "px_fb": 0, "t_meta": 0.0, "t_px": 0.0,
       "t_order": 0.0, "n_order": 0}
_FD_LOCK = _fd_threading.Lock()


def _fd_note(kind, fast, dt):
    with _FD_LOCK:
        _FD[kind + ("_fast" if fast else "_fb")] += 1
        _FD["t_" + kind] += dt


@_fd_atexit.register
def _fd_report():
    print("[fastdcm] meta fast=%d fallback=%d | px fast=%d fallback=%d | cum: meta %.1fs px %.1fs"
          " | ORDER-PASS WALL %.2fs over %d series | threads=%s"
          % (_FD["meta_fast"], _FD["meta_fb"], _FD["px_fast"], _FD["px_fb"],
             _FD["t_meta"], _FD["t_px"], _FD["t_order"], _FD["n_order"],
             os.environ.get("ARM_RD_THREADS", "16")), flush=True)


In [ ]:
# COATNET_TRANSFORMER_BLEND_V1
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
IMG = 336
CROP_MM = 140.0
# 64 slices per study instead of 44, same proportions. Must match the corpus the weights
# were trained on (knee_corpus_v4.py).
SLOTS = [("Sagittal", 1, 27), ("Sagittal", 0, 21), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 64
K_EVAL = 48   # every window position the volume holds, not an evenly spaced subset
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [
    {"file": "raptor_ft_EFFV2S_288.pt", "arch": "tf_efficientnetv2_s.in21k_ft_in1k", "res": 288, "norm": "imagenet", "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(device)
    use_cuda = device != "cpu" and str(device).startswith("cuda")
    if use_cuda:
        # fp16 conv on T4 is fully cuDNN-supported (bf16 is NOT -> "no engine").
        try:
            with torch.autocast("cuda", dtype=torch.float16):
                o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
        except RuntimeError:
            # fp32 always has a Turing conv engine; slower but never drops a study.
            torch.cuda.empty_cache()
            o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
    o = torch.sigmoid(model(x).float())
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from concurrent.futures import ThreadPoolExecutor as _ArmPool
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    _RD_THREADS = int(os.environ.get("ARM_RD_THREADS", "16"))

    def _hdr_one_pydicom(f):
        """One header -> (pos, file, pixel_spacing, ok). Identical arithmetic to the serial loop."""
        try:
            # Parse ONLY the four elements used below. Measured 7.4x faster than a
            # full header parse over the same files, with identical ordering.
            h = pydicom.dcmread(f, stop_before_pixels=True, force=True,
                                specific_tags=[(0x0020, 0x0032), (0x0020, 0x0037),
                                               (0x0028, 0x0030), (0x0020, 0x0013)])
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def _hdr_one(f):
        """One header -> (pos, file, pixel_spacing, ok). Identical arithmetic to the serial loop."""
        try:
            # Direct explicit-VR-LE scan of the same four elements. The competition files are all
            # uncompressed explicit VR little endian, so the tags can be taken from a single 4 KB
            # pread. Verified byte-identical to the pydicom parse (ordering key, pixel spacing,
            # ok-flag) over a real series; falls back to pydicom for anything it declines.
            _t0 = _fd_clock()
            m = _fd_read_meta(f)
            if m is None:
                _fd_note('meta', False, _fd_clock() - _t0)
                return _hdr_one_pydicom(f)
            iop, ipp = m.iop, m.ipp
            if iop and ipp and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(m.inst or 0)
            ps = float(m.spacing[0]) if m.spacing else 0.5
            _fd_note('meta', True, _fd_clock() - _t0)
            return (pos, f, ps, True)
        except Exception:
            return _hdr_one_pydicom(f)

    def read_px(f):
        # PixelData is the final element and uncompressed, so the pixels are the file's tail and
        # come back with one seek+read. When slope==1 and intercept==0 the float64 multiply inside
        # apply_modality_lut is the identity and uint16 -> float32 is exact (65535 < 2**24), so the
        # array is the same one, built with fewer copies. Anything unusual falls back to pydicom.
        try:
            _t0 = _fd_clock()
            # one open serves both: the header is the head of the same buffer the pixels end
            m, a = _fd_read_one(f)
            if m is None or a is None or m.has_modlut:
                _fd_note('px', False, _fd_clock() - _t0)
                return _read_px_pydicom(f)
            if m.has_rescale and not (m.slope == 1.0 and m.inter == 0.0):
                a = (a.astype(np.float64) * m.slope + m.inter).astype(np.float32)
            else:
                a = a.astype(np.float32)
            if m.photo == 'MONOCHROME1':
                a = a.max() - a
            _fd_note('px', True, _fd_clock() - _t0)
            return a
        except Exception:
            return _read_px_pydicom(f)

    def order_and_meta(sdir):
        # Threaded because the read is IO-bound and was the single largest cost in the run.
        # pool.map preserves INPUT order, so the stable sort below breaks ties exactly as the
        # serial loop did -> byte-identical slice ordering.
        fs = glob.glob(sdir + "/*.dcm")
        if not fs:
            return [], 0.5
        if len(fs) > 1 and _RD_THREADS > 1:
            with _ArmPool(max_workers=min(_RD_THREADS, len(fs))) as pool:
                out = list(pool.map(_hdr_one, fs))
        else:
            out = [_hdr_one(f) for f in fs]
        recs = [(p, f, ps) for p, f, ps, _ok in out]
        ps_list = [ps for _p, _f, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps
    def _read_px_pydicom(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    def _timed_order(sdir):
        # wall clock, not cumulative: this is the number that moves when threads change
        _t0 = _fd_clock()
        try:
            return order_and_meta(sdir)
        finally:
            with _FD_LOCK:
                _FD["t_order"] += _fd_clock() - _t0
                _FD["n_order"] += 1

    return _timed_order, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (knee_corpus_v2.py, SPAN_LO/SPAN_HI).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        from concurrent.futures import ThreadPoolExecutor as _PxPool
        def _px_one(_p):
            _fp, _ps = files[min(_p, n - 1)]
            try:
                return (read_px(_fp), _ps)
            except Exception:
                return (None, med_ps)
        if len(picks) > 1:
            with _PxPool(max_workers=min(8, len(picks))) as _px_ex:
                _got = list(_px_ex.map(_px_one, list(picks)))
        else:
            _got = [_px_one(_p) for _p in picks]
        arrs = [g[0] for g in _got]; pss = [g[1] for g in _got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask


# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower():
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    from concurrent.futures import ThreadPoolExecutor as _ArmPool2
    reader = _make_reader()
    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]

    # SEQUENTIAL ARMS (the OOM fix): only ONE model is resident at a time, so peak system RAM ==
    # one model == the single-arm champion's footprint (which graded fine at 0.879). Holding both
    # arms simultaneously OOM'd system RAM on the full hidden test. Each study is re-preprocessed
    # per arm (build_study is cheap vs inference) and every per-study buffer is freed. Same models,
    # same windowing, same rank-mean blend -> identical 0.8893 result, just serialized.
    # SINGLE PASS over the test set. Every arm here is the same architecture at the same
    # resolution, so eval_windows() returns the IDENTICAL tensor for each of them. The original
    # loop was arms-outer, which re-read and re-preprocessed all 1,322 studies once PER ARM, and
    # DICOM reading is the whole wall clock (header scan alone was 72.9% of IO). Building each
    # study once and running both models on the result halves the run. Numerically this is the
    # same computation: same xw, same infer_probs, same arm_probs, so the blend is unchanged.
    _NG = torch.cuda.device_count() if str(dev).startswith("cuda") else 0
    # One arm per GPU, resident. No DataParallel, so no per-forward module replication.
    _arm_devs = [(f"cuda:{a % _NG}" if _NG > 0 else dev) for a in range(len(ARMS))]
    _sp_models = []
    for a, arm in enumerate(ARMS):
        wp = find_weight_file(arm["file"])
        _m, _res = load_model(wp, arm["arch"], arm["res"], _arm_devs[a])
        _sp_models.append((_m, _res, arm.get("norm", "imagenet")))
        # Identify the exact weight file that was mounted. A dataset version that is still
        # processing when the kernel starts silently serves the PREVIOUS version, and the run
        # then looks correct while scoring an older checkpoint. Printing size and digest makes
        # that visible in the log instead of leaving it to be inferred from timestamps.
        _wsz = os.path.getsize(wp)
        _h = hashlib.md5()
        with open(wp, "rb") as _f:                     # chunked: the largest arm is 670 MB and
            for _chunk in iter(lambda: _f.read(1 << 22), b""):   # two models are resident here
                _h.update(_chunk)
        _wmd = _h.hexdigest()[:12]
        print(f"[arm {a}] loaded {arm['file']} | res {_res} | dev {_arm_devs[a]} | {_wsz} bytes | md5 {_wmd} | {time.time()-t0:.0f}s", flush=True)
    # Arms may now differ in RESOLUTION. Scale turns out to decorrelate arms as well as architecture
    # does - a ConvNeXt at 224 disagrees with our CoAtNet champion more than a Swin transformer does -
    # but the old single-pass path asserted one resolution across arms, so that whole axis was
    # unshippable. Building one window tensor per DISTINCT resolution costs only the resize: the DICOM
    # decode, which is the entire wall clock, is done once per study and shared.
    _RN_SET = sorted({(_r, _n) for _, _r, _n in _sp_models})
    _SP_RES = _RN_SET[0][0]
    print(f"[res/norm] arms at {_RN_SET}", flush=True)
    _PARALLEL_ARMS = _NG > 1 and len(_sp_models) > 1
    # ONE pool for the whole run. Creating it per study spawned and tore down threads 1,322 times
    # and cost more than it saved (measured 27s -> 49s on the dry run before this was hoisted).
    _arm_ex = _ArmPool2(max_workers=len(_sp_models)) if _PARALLEL_ARMS else None
    print(f"[gpu] {_NG} visible | arms run {'CONCURRENTLY, one per GPU' if _PARALLEL_ARMS else 'sequentially'}", flush=True)

    _PF = int(os.environ.get("ARM_PREFETCH", "2"))

    def _prep_study(sid):
        vol, mask = build_study(sid, SER, tsdir, reader)
        xws = {(_r, _n): eval_windows(vol, mask, k=K_EVAL, res=_r, norm=_n) for (_r, _n) in _RN_SET}
        del vol, mask
        return xws

    # Producer/consumer: the pool prepares study i+1.._PF while the GPU consumes study i.
    # Consumption order is still strictly 0..N-1, so arm_probs is filled exactly as before.
    with _ArmPool2(max_workers=max(1, _PF)) as _pf_ex:
        _fut = {}
        for _j in range(min(_PF + 1, N)):
            _fut[_j] = _pf_ex.submit(_prep_study, test_ids[_j])
        for i, sid in enumerate(test_ids):
            try:
                xw = _fut.pop(i).result()
                if _PARALLEL_ARMS:
                    # Both arms at once, each on its own device. infer_probs moves the tensor.
                    def _run_arm(_t):
                        _a, (_mm, _rr, _nn) = _t
                        return _a, infer_probs(_mm, xw[(_rr, _nn)], _arm_devs[_a])
                    for _a, _o in _arm_ex.map(_run_arm, list(enumerate(_sp_models))):
                        arm_probs[_a][i] = _o
                else:
                    for a, (_m, _r, _n) in enumerate(_sp_models):
                        arm_probs[a][i] = infer_probs(_m, xw[(_r, _n)], _arm_devs[a])
                del xw
            except Exception as e:
                _fut.pop(i, None)
                print(f"  study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            _nx = i + _PF + 1
            if _nx < N:
                _fut[_nx] = _pf_ex.submit(_prep_study, test_ids[_nx])
            if (i + 1) % 100 == 0 or i + 1 == N:
                print(f"  [1pass] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    del _sp_models
    gc.collect()
    if str(dev).startswith("cuda"):
        torch.cuda.empty_cache()
    print(f"[1pass] all arms done + freed | {time.time()-t0:.0f}s", flush=True)

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission_coatnet.csv"
    sub.to_csv(out, index=False)
    sub.to_csv("/kaggle/working/submission.csv", index=False)
    print("ONEPASS: wrote submission.csv", flush=True)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as _coat_exc:
        import traceback as _coat_traceback
        print(f"CoAtNet branch failed; retaining transformer submission: {type(_coat_exc).__name__}: {_coat_exc}", flush=True)
        _coat_traceback.print_exc()



# Blend two independently validated rank predictors. The default remains the transformer
# submission if the CoAtNet branch did not complete, so a recoverable branch failure
# cannot erase a valid competition artifact.
from pathlib import Path as _BlendPath
import numpy as _blend_np
import pandas as _blend_pd

_blend_work = _BlendPath('/kaggle/working')
_blend_transformer_path = _blend_work / 'submission.csv'
_blend_coatnet_path = _blend_work / 'submission_coatnet.csv'
if _blend_coatnet_path.is_file():
    _blend_transformer = _blend_pd.read_csv(_blend_transformer_path, dtype={'StudyInstanceUID': str})
    _blend_coatnet = _blend_pd.read_csv(_blend_coatnet_path, dtype={'StudyInstanceUID': str})
    _blend_labels = [c for c in _blend_transformer.columns if c != 'StudyInstanceUID']
    if _blend_coatnet.columns.tolist() != _blend_transformer.columns.tolist():
        raise RuntimeError('CoAtNet/transformer submission schema mismatch')
    if _blend_coatnet['StudyInstanceUID'].tolist() != _blend_transformer['StudyInstanceUID'].tolist():
        raise RuntimeError('CoAtNet/transformer study order mismatch')
    _blend_tr = _blend_transformer[
        _blend_labels
    ].rank(
        method='average',
        pct=True,
    )

    _blend_cr = _blend_coatnet[
        _blend_labels
    ].rank(
        method='average',
        pct=True,
    )

    _blend_output = (
        _blend_transformer.copy()
    )

    _coatnet_weight = {
        label: 0.55
        for label in _blend_labels
    }

    if globals().get(
        'V18_CALIBRATOR_APPLIED',
        False,
    ):
        _coatnet_weight.update(
            {
                'Medial Meniscus': 0.57,
                'Lateral Meniscus': 0.59,
                'Fracture': 0.59,
            }
        )

    for _label in _blend_labels:
        _cw = float(
            _coatnet_weight[
                _label
            ]
        )

        _blend_output[
            _label
        ] = (
            (
                1.0
                - _cw
            )
            * _blend_tr[
                _label
            ]
            + _cw
            * _blend_cr[
                _label
            ]
        )

    _blend_output[
        _blend_labels
    ] = _blend_output[
        _blend_labels
    ].rank(
        method='average',
        pct=True,
    )

    print(
        '[V18] CoAtNet target weights: '
        + ', '.join(
            f'{label}='
            f'{_coatnet_weight[label]:.2f}'
            for label in _blend_labels
            if (
                _coatnet_weight[label]
                != 0.50
            )
        ),
        flush=True,
    )

    _blend_values = _blend_output[
        _blend_labels
    ].to_numpy(
        _blend_np.float64
    )
    if not _blend_np.isfinite(_blend_values).all() or _blend_values.min() < 0 or _blend_values.max() > 1:
        raise RuntimeError('invalid blended prediction values')
    _blend_output.to_csv(_blend_transformer_path, index=False)
    print(f'final submission.csv = CoAtNet 384 two scratch-trained arms, rank blend; {_blend_output.shape}', flush=True)
else:
    print('CoAtNet output unavailable; submission.csv remains the validated transformer ensemble', flush=True)

# V18 output hygiene.
for _v18_temp in (
    _blend_work / 'submission_coatnet.csv',
    _blend_work / 'submission_transformer_0920.csv',
):
    try:
        if _v18_temp.is_file():
            _v18_temp.unlink()
    except OSError:
        pass


In [ ]:
_fd_report()
